# Train model candidates on Google Colab

Notebook này chỉ điều phối code đã kiểm thử trong repository. Dữ liệu được xác minh SHA-256, copy từ Google Drive sang `/content` để train, còn manifest/checkpoint/metrics được lưu về Drive. Chạy cả hai candidate trên cùng dataset/split để so sánh; tuyệt đối không chạy notebook này trên laptop.

In [ ]:
REPO_URL = "https://github.com/DangHung1110/IoT-CK.git"
GIT_REF = "feat/model-training-pipeline"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/parcel_sorter"  # @param {type:"string"}
DATASET_VERSION = "v1"  # @param {type:"string"}
ARCHIVE_RELATIVE_PATH = "data/archives/dataset_v1_splits.zip"  # @param {type:"string"}
MANIFEST_RELATIVE_PATH = "data/manifests/dataset_v1.csv"  # @param {type:"string"}
DATASET_ARCHIVE_SHA256 = "REPLACE_WITH_64_LOWERCASE_HEX"  # @param {type:"string"}
DATASET_MANIFEST_SHA256 = "REPLACE_WITH_64_LOWERCASE_HEX"  # @param {type:"string"}
CANDIDATES = "mobilenet_v3_small,yolov8n_cls"  # @param {type:"string"}
SEED = 42  # @param {type:"integer"}
BATCH_SIZE = 32  # @param {type:"integer"}
RESUME_RUN_ID = None
RESUME_CHECKPOINT_RELATIVE_PATH = None

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import re
import subprocess
from pathlib import Path

repo_dir = Path("/content/IoT-CK")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(repo_dir)],
        check=True,
    )
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "--prune"], check=True)
target_ref = GIT_REF if re.fullmatch(r"[0-9a-f]{40}", GIT_REF) else f"origin/{GIT_REF}"
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", target_ref], check=True)

In [ ]:
import sys

if not ((3, 11) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError(f"Unsupported Colab Python: {sys.version.split()[0]}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir), "--no-deps"],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "-r",
        str(repo_dir / "requirements" / "colab-training-overlay.txt"),
    ],
    check=True,
)

In [ ]:
from importlib import metadata

from parcel_sorter.ml.contracts import ModelCandidate
from parcel_sorter.ml.yolo_training import ULTRALYTICS_VERSION

for label, digest in {
    "archive": DATASET_ARCHIVE_SHA256,
    "manifest": DATASET_MANIFEST_SHA256,
}.items():
    if not re.fullmatch(r"[0-9a-f]{64}", digest) or digest == "0" * 64:
        raise ValueError(f"Replace {label} SHA-256 with the verified digest")
candidate_values = tuple(item.strip() for item in CANDIDATES.split(",") if item.strip())
candidates = tuple(ModelCandidate(item) for item in candidate_values)
if len(candidates) != len(set(candidates)) or not candidates:
    raise ValueError("CANDIDATES must contain unique supported candidates")
if ModelCandidate.YOLOV8N_CLS in candidates:
    actual_ultralytics = metadata.version("ultralytics")
    if actual_ultralytics != ULTRALYTICS_VERSION:
        raise RuntimeError(
            f"Expected ultralytics {ULTRALYTICS_VERSION}, found {actual_ultralytics}"
        )
print(f"Candidates: {[item.value for item in candidates]}")

In [ ]:
from parcel_sorter.data import ParcelClass
from parcel_sorter.ml.colab_training import train_candidate_on_colab
from parcel_sorter.ml.contracts import (
    AugmentationContract,
    ExperimentConfig,
    PreprocessContract,
)

class_order = tuple(sorted(ParcelClass, key=lambda item: item.value))
results = {}
for candidate in candidates:
    is_mobilenet = candidate is ModelCandidate.MOBILENET_V3_SMALL
    config = ExperimentConfig(
        candidate=candidate,
        dataset_version=DATASET_VERSION,
        dataset_manifest_sha256=DATASET_MANIFEST_SHA256,
        dataset_archive_sha256=DATASET_ARCHIVE_SHA256,
        split_seed=SEED,
        seed=SEED,
        classes=class_order,
        preprocess=PreprocessContract(
            preprocess_id=(
                "mobilenet-rgb320-imagenet-v1" if is_mobilenet else "yolov8-rgb320-scale-v1"
            ),
            resize="resize_square",
            mean=(0.485, 0.456, 0.406) if is_mobilenet else (0.0, 0.0, 0.0),
            std=(0.229, 0.224, 0.225) if is_mobilenet else (1.0, 1.0, 1.0),
        ),
        augmentation=AugmentationContract(),
        pretrained_checkpoint=(
            "MobileNet_V3_Small_Weights.IMAGENET1K_V1" if is_mobilenet else "yolov8n-cls.pt"
        ),
        batch_size=BATCH_SIZE,
    )
    run_dir, best_checkpoint = train_candidate_on_colab(
        config=config,
        repo_dir=repo_dir,
        drive_root=DRIVE_ROOT,
        archive_relative_path=ARCHIVE_RELATIVE_PATH,
        manifest_relative_path=MANIFEST_RELATIVE_PATH,
        local_workspace=Path(f"/content/parcel_sorter_train/{candidate.value}"),
        resume_run_id=RESUME_RUN_ID,
        resume_checkpoint_relative_path=RESUME_CHECKPOINT_RELATIVE_PATH,
    )
    results[candidate.value] = {
        "run_dir": run_dir.as_posix(),
        "best_checkpoint": best_checkpoint.as_posix(),
    }
results